In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
print(os.listdir("/content/drive/MyDrive"))

['Colab Notebooks', 'faces.zip']


In [3]:
!rm -rf /content/faces

In [4]:
!unzip -q -o "/content/drive/MyDrive/faces.zip" -d "/content/"

In [5]:
import os
print(os.listdir("/content/faces"))


['real', 'fake']


In [6]:
import torch
import torch.nn as nn
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import os

In [7]:
class FaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        for label_name, label_value in [("real", 0), ("fake", 1)]:
            folder = os.path.join(root_dir, label_name)
            for video_folder in os.listdir(folder):
                video_path = os.path.join(folder, video_folder)
                for img_file in os.listdir(video_path):
                    img_path = os.path.join(video_path, img_file)
                    self.samples.append((img_path, label_value))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

In [8]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [9]:
full_dataset = FaceDataset("/content/faces", transform=transform)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")

Training images: 3070
Validation images: 768


In [10]:
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 110MB/s] 


In [11]:
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 2)

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Using device: {device}")

Using device: cuda


In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [15]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")

Epoch 1/10 - Loss: 0.3755
Epoch 2/10 - Loss: 0.1422
Epoch 3/10 - Loss: 0.0795
Epoch 4/10 - Loss: 0.0526
Epoch 5/10 - Loss: 0.0309
Epoch 6/10 - Loss: 0.0221
Epoch 7/10 - Loss: 0.0219
Epoch 8/10 - Loss: 0.0197
Epoch 9/10 - Loss: 0.0188
Epoch 10/10 - Loss: 0.0242


In [16]:
model.eval()
correct, total = 0, 0
real_correct, real_total = 0, 0
fake_correct, fake_total = 0, 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        real_mask = labels == 0
        fake_mask = labels == 1
        real_total += real_mask.sum().item()
        fake_total += fake_mask.sum().item()
        real_correct += ((predicted == labels) & real_mask).sum().item()
        fake_correct += ((predicted == labels) & fake_mask).sum().item()

print(f"Overall accuracy: {100 * correct / total:.2f}%")
print(f"Real faces correct: {100 * real_correct / max(real_total, 1):.2f}%  ({real_total} images)")
print(f"Fake faces correct: {100 * fake_correct / max(fake_total, 1):.2f}%  ({fake_total} images)")

Overall accuracy: 96.48%
Real faces correct: 95.57%  (158 images)
Fake faces correct: 96.72%  (610 images)


In [17]:
import random
from torch.utils.data import Subset

random.seed(42)

def get_videos(label_name):
    folder = os.path.join("/content/faces", label_name)
    return [os.path.join(folder, v) for v in os.listdir(folder)]

real_videos = get_videos("real")
fake_videos = get_videos("fake")
random.shuffle(real_videos)
random.shuffle(fake_videos)

def split(vids, ratio=0.8):
    n = int(ratio * len(vids))
    return vids[:n], vids[n:]

real_train, real_val = split(real_videos)
fake_train, fake_val = split(fake_videos)

train_videos = set(real_train + fake_train)
val_videos = set(real_val + fake_val)

train_indices = [i for i, (p, _) in enumerate(full_dataset.samples) if os.path.dirname(p) in train_videos]
val_indices = [i for i, (p, _) in enumerate(full_dataset.samples) if os.path.dirname(p) in val_videos]

train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Train videos: {len(train_videos)} | Val videos: {len(val_videos)}")
print(f"Train images: {len(train_dataset)} | Val images: {len(val_dataset)}")

Train videos: 319 | Val videos: 81
Train images: 3053 | Val images: 785


In [18]:
model = models.efficientnet_b0(weights='IMAGENET1K_V1')
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [19]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")

Epoch 1/10 - Loss: 0.3777
Epoch 2/10 - Loss: 0.1424
Epoch 3/10 - Loss: 0.0791
Epoch 4/10 - Loss: 0.0521
Epoch 5/10 - Loss: 0.0284
Epoch 6/10 - Loss: 0.0265
Epoch 7/10 - Loss: 0.0177
Epoch 8/10 - Loss: 0.0127
Epoch 9/10 - Loss: 0.0173
Epoch 10/10 - Loss: 0.0200


In [20]:
model.eval()
correct, total = 0, 0
real_correct, real_total = 0, 0
fake_correct, fake_total = 0, 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        real_mask = labels == 0
        fake_mask = labels == 1
        real_total += real_mask.sum().item()
        fake_total += fake_mask.sum().item()
        real_correct += ((predicted == labels) & real_mask).sum().item()
        fake_correct += ((predicted == labels) & fake_mask).sum().item()

print(f"Overall accuracy: {100 * correct / total:.2f}%")
print(f"Real faces correct: {100 * real_correct / max(real_total, 1):.2f}%  ({real_total} images)")
print(f"Fake faces correct: {100 * fake_correct / max(fake_total, 1):.2f}%  ({fake_total} images)")

Overall accuracy: 93.25%
Real faces correct: 82.89%  (152 images)
Fake faces correct: 95.73%  (633 images)


In [21]:
torch.save(model.state_dict(), "/content/drive/MyDrive/visual_model.pth")
print("Model saved to Google Drive.")

Model saved to Google Drive.


In [2]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

# re-extract the faces only if the session was wiped
if not os.path.exists("/content/faces"):
    subprocess.run(["unzip", "-q", "-o", "/content/drive/MyDrive/faces.zip", "-d", "/content/"], check=True)
print("Folders:", os.listdir("/content/faces"))

import torch
import torch.nn as nn
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image

class FaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        for label_name, label_value in [("real", 0), ("fake", 1)]:
            folder = os.path.join(root_dir, label_name)
            for video_folder in os.listdir(folder):
                video_path = os.path.join(folder, video_folder)
                for img_file in os.listdir(video_path):
                    self.samples.append((os.path.join(video_path, img_file), label_value))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
full_dataset = FaceDataset("/content/faces", transform=transform)

print("Device:", device)
print("Total images:", len(full_dataset))

Mounted at /content/drive
Folders: ['real', 'fake']
Device: cuda
Total images: 3838


In [3]:
import json, random, copy
import numpy as np, pandas as pd
from torch.utils.data import Subset
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, confusion_matrix
import torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)

with open("/content/drive/MyDrive/metadata.json") as f:
    meta = json.load(f)

def vid_id(name): return name.replace(".mp4", "")
label_of = {vid_id(n): (0 if i["label"] == "REAL" else 1) for n, i in meta.items()}

# group = a real video plus all fakes made from it
groups = {}
for name, info in meta.items():
    vid = vid_id(name)
    g = vid_id(info["original"]) if (info["label"] == "FAKE" and info.get("original")) else vid
    groups.setdefault(g, []).append(vid)

with_real = [g for g, m in groups.items() if any(label_of[v] == 0 for v in m)]
no_real   = [g for g, m in groups.items() if not any(label_of[v] == 0 for v in m)]

def three_way(gs, tr=0.70, va=0.15, seed=42):
    gs = sorted(gs)
    random.Random(seed).shuffle(gs)
    a = int(tr * len(gs)); b = int((tr + va) * len(gs))
    return gs[:a], gs[a:b], gs[b:]

split = {"train": [], "val": [], "test": []}
for gs in (with_real, no_real):
    tr, va, te = three_way(gs)
    for name, chosen in (("train", tr), ("val", va), ("test", te)):
        for g in chosen:
            split[name].extend(groups[g])

s = {k: set(v) for k, v in split.items()}
assert not (s["train"] & s["val"]) and not (s["train"] & s["test"]) and not (s["val"] & s["test"])
assert sum(len(v) for v in split.values()) == len(meta)

for k, v in split.items():
    r = sum(label_of[x] == 0 for x in v)
    print(f"{k:5s}: {len(v)} videos | real {r} | fake {len(v) - r}")

with open("/content/drive/MyDrive/split_v1.json", "w") as f:
    json.dump(split, f)
print("Saved split_v1.json to Drive")

train: 292 videos | real 53 | fake 239
val  : 54 videos | real 12 | fake 42
test : 54 videos | real 12 | fake 42
Saved split_v1.json to Drive


In [4]:
def video_of(path):
    return os.path.basename(os.path.dirname(path))

def make_subset(vids):
    vids = set(vids)
    idx = [i for i, (p, _) in enumerate(full_dataset.samples) if video_of(p) in vids]
    return Subset(full_dataset, idx)

train_dataset = make_subset(split["train"])
val_dataset   = make_subset(split["val"])
test_dataset  = make_subset(split["test"])

assert len(train_dataset) + len(val_dataset) + len(test_dataset) == len(full_dataset), "Some images were not assigned"

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

for name, ds in [("Train", train_dataset), ("Val", val_dataset), ("Test", test_dataset)]:
    labs = [full_dataset.samples[i][1] for i in ds.indices]
    print(f"{name}: {len(ds)} images | real {labs.count(0)} | fake {labs.count(1)}")

Train: 2786 images | real 522 | fake 2264
Val: 514 images | real 120 | fake 394
Test: 538 images | real 119 | fake 419


In [5]:
train_labels = [full_dataset.samples[i][1] for i in train_dataset.indices]
n_real, n_fake = train_labels.count(0), train_labels.count(1)
total = n_real + n_fake
class_weights = torch.tensor([total / (2 * n_real), total / (2 * n_fake)], dtype=torch.float32).to(device)
print(f"Class weights -> real: {class_weights[0].item():.3f} | fake: {class_weights[1].item():.3f}")

set_seed(42)
model = models.efficientnet_b0(weights='IMAGENET1K_V1')
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

Class weights -> real: 2.669 | fake: 0.615
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 162MB/s]


In [6]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, labels = [], []
    for images, y in loader:
        out = model(images.to(device))
        probs.extend(torch.softmax(out, dim=1)[:, 1].cpu().tolist())
        labels.extend(y.tolist())
    return np.array(probs), np.array(labels)

def video_level(probs, labels, subset):
    vids = [video_of(full_dataset.samples[i][0]) for i in subset.indices]
    df = pd.DataFrame({"video": vids, "fake_prob": probs, "label": labels})
    return df.groupby("video").agg(fake_prob=("fake_prob", "mean"), label=("label", "first")).reset_index()

def summarize(probs, labels, title):
    preds = (probs >= 0.5).astype(int)
    print(title)
    print(f"  Balanced accuracy: {balanced_accuracy_score(labels, preds) * 100:.2f}%")
    print(f"  Real correct: {(preds[labels == 0] == 0).mean() * 100:.2f}% | Fake correct: {(preds[labels == 1] == 1).mean() * 100:.2f}%")
    print(f"  AUC: {roc_auc_score(labels, probs):.4f}")
    print(f"  Confusion matrix (rows = true real/fake, cols = predicted real/fake):\n{confusion_matrix(labels, preds)}")

In [7]:
num_epochs = 10
best_bal, best_epoch, best_state = 0.0, 0, None

for epoch in range(num_epochs):
    model.train()
    running = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running += loss.item()

    probs, labels_np = predict(model, val_loader)
    bal = balanced_accuracy_score(labels_np, (probs >= 0.5).astype(int))
    print(f"Epoch {epoch+1}/{num_epochs} | train loss {running/len(train_loader):.4f} | val balanced acc {bal*100:.2f}%")

    if bal > best_bal:
        best_bal, best_epoch = bal, epoch + 1
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, "/content/drive/MyDrive/visual_model_v2.pth")

model.load_state_dict(best_state)
print(f"\nBest epoch: {best_epoch} (val balanced acc {best_bal*100:.2f}%). Model restored to that epoch.")

Epoch 1/10 | train loss 0.5255 | val balanced acc 84.40%
Epoch 2/10 | train loss 0.2201 | val balanced acc 90.27%
Epoch 3/10 | train loss 0.1156 | val balanced acc 91.76%
Epoch 4/10 | train loss 0.1072 | val balanced acc 92.72%
Epoch 5/10 | train loss 0.0686 | val balanced acc 91.34%
Epoch 6/10 | train loss 0.0827 | val balanced acc 90.90%
Epoch 7/10 | train loss 0.0611 | val balanced acc 91.59%
Epoch 8/10 | train loss 0.0339 | val balanced acc 91.92%
Epoch 9/10 | train loss 0.0373 | val balanced acc 91.05%
Epoch 10/10 | train loss 0.0319 | val balanced acc 91.47%

Best epoch: 4 (val balanced acc 92.72%). Model restored to that epoch.


In [8]:
val_probs, val_labels = predict(model, val_loader)
summarize(val_probs, val_labels, "VALIDATION - per image")
val_videos = video_level(val_probs, val_labels, val_dataset)
summarize(val_videos["fake_prob"].values, val_videos["label"].values, "VALIDATION - per video")

VALIDATION - per image
  Balanced accuracy: 92.72%
  Real correct: 90.00% | Fake correct: 95.43%
  AUC: 0.9534
  Confusion matrix (rows = true real/fake, cols = predicted real/fake):
[[108  12]
 [ 18 376]]
VALIDATION - per video
  Balanced accuracy: 94.64%
  Real correct: 91.67% | Fake correct: 97.62%
  AUC: 0.9544
  Confusion matrix (rows = true real/fake, cols = predicted real/fake):
[[11  1]
 [ 1 41]]


In [9]:
test_probs, test_labels = predict(model, test_loader)
summarize(test_probs, test_labels, "TEST - per image")
test_videos = video_level(test_probs, test_labels, test_dataset)
summarize(test_videos["fake_prob"].values, test_videos["label"].values, "TEST - per video")

TEST - per image
  Balanced accuracy: 79.02%
  Real correct: 66.39% | Fake correct: 91.65%
  AUC: 0.8794
  Confusion matrix (rows = true real/fake, cols = predicted real/fake):
[[ 79  40]
 [ 35 384]]
TEST - per video
  Balanced accuracy: 89.29%
  Real correct: 83.33% | Fake correct: 95.24%
  AUC: 0.9147
  Confusion matrix (rows = true real/fake, cols = predicted real/fake):
[[10  2]
 [ 2 40]]


In [10]:
val_videos.to_csv("/content/drive/MyDrive/visual_val_video_scores.csv", index=False)
test_videos.to_csv("/content/drive/MyDrive/visual_test_video_scores.csv", index=False)

vv, vl = val_videos["fake_prob"].values, val_videos["label"].values
tv, tl = test_videos["fake_prob"].values, test_videos["label"].values

info = {
    "model": "efficientnet_b0 (ImageNet pretrained), 2-class head",
    "split_file": "split_v1.json",
    "input_size": 224,
    "normalize_mean": [0.485, 0.456, 0.406],
    "normalize_std": [0.229, 0.224, 0.225],
    "label_map": {"real": 0, "fake": 1},
    "class_weights": [class_weights[0].item(), class_weights[1].item()],
    "learning_rate": 0.0001, "batch_size": 32, "seed": 42,
    "epochs_run": num_epochs, "best_epoch": best_epoch,
    "val_video_balanced_acc": float(balanced_accuracy_score(vl, (vv >= 0.5).astype(int))),
    "val_video_auc": float(roc_auc_score(vl, vv)),
    "test_video_balanced_acc": float(balanced_accuracy_score(tl, (tv >= 0.5).astype(int))),
    "test_video_auc": float(roc_auc_score(tl, tv)),
}
with open("/content/drive/MyDrive/visual_model_v2_info.json", "w") as f:
    json.dump(info, f, indent=2)
print(json.dumps(info, indent=2))

{
  "model": "efficientnet_b0 (ImageNet pretrained), 2-class head",
  "split_file": "split_v1.json",
  "input_size": 224,
  "normalize_mean": [
    0.485,
    0.456,
    0.406
  ],
  "normalize_std": [
    0.229,
    0.224,
    0.225
  ],
  "label_map": {
    "real": 0,
    "fake": 1
  },
  "class_weights": [
    2.6685824394226074,
    0.6152827143669128
  ],
  "learning_rate": 0.0001,
  "batch_size": 32,
  "seed": 42,
  "epochs_run": 10,
  "best_epoch": 4,
  "val_video_balanced_acc": 0.9464285714285714,
  "val_video_auc": 0.9543650793650793,
  "test_video_balanced_acc": 0.8928571428571428,
  "test_video_auc": 0.9146825396825397
}
